# Hypergraph Centrality: Eight-Actor Social Example
This notebook computes hyperdegree, closeness, eigenvector centrality, betweenness, Hypergraph Dangling Centrality, runtime, and complexity for the validated $8\times5$ hypergraph.


In [ ]:
"""Eight-person social hypergraph: centralities, dangling impact, and timings.

The implementation uses only NumPy and the Python standard library.
Unreachable distances are stored as infinity; diagonal distances are zero.
"""

from __future__ import annotations

from collections import deque
from itertools import combinations
from statistics import median
from time import perf_counter
import csv

import numpy as np


# Meaningful social-network interpretation of the original 8-node hypergraph.
PEOPLE = {
    1: "Community coordinator",
    2: "Data analyst",
    3: "Content creator",
    4: "Event organizer",
    5: "Cross-group liaison",
    6: "Volunteer lead",
    7: "Student representative",
    8: "Outreach officer",
}

GROUPS = {
    "e1": ("Research discussion group", {1, 2, 3}),
    "e2": ("Community project team", {2, 3, 4, 5}),
    "e3": ("Event coordination group", {4, 5, 6}),
    "e4": ("Community outreach group", {5, 6, 7, 8}),
    "e5": ("Student support group", {1, 7, 8}),
}


def incidence_matrix(vertices, groups):
    vertices = list(vertices)
    edge_sets = [members for _, members in groups.values()]
    index = {v: i for i, v in enumerate(vertices)}
    H = np.zeros((len(vertices), len(edge_sets)), dtype=int)
    for j, edge in enumerate(edge_sets):
        for v in edge:
            H[index[v], j] = 1
    return H


def projected_adjacency(H, weighted=False):
    """Vertex projection; diagonal incidence counts are removed."""
    A = H @ H.T
    np.fill_diagonal(A, 0)
    return A.astype(float) if weighted else (A > 0).astype(int)


def adjacency_lists(A):
    return {i: set(np.flatnonzero(A[i])) for i in range(len(A))}


def distance_matrix(A):
    """Unweighted all-pairs distances; infinity denotes no path."""
    n = len(A)
    neighbors = adjacency_lists(A)
    D = np.full((n, n), np.inf)
    np.fill_diagonal(D, 0.0)
    for source in range(n):
        queue = deque([source])
        while queue:
            u = queue.popleft()
            for v in neighbors[u]:
                if np.isinf(D[source, v]):
                    D[source, v] = D[source, u] + 1
                    queue.append(v)
    return D


def communication_strength(D):
    """Sum reciprocal distances over unique pairs i < j; 1/inf = 0."""
    upper = D[np.triu_indices(len(D), k=1)]
    return float(np.sum(np.where(np.isfinite(upper), 1.0 / upper, 0.0)))


def hyperdegree(H):
    return H.sum(axis=1).astype(float)


def closeness(A):
    D = distance_matrix(A)
    n = len(A)
    scores = np.zeros(n)
    for i in range(n):
        reachable = D[i][np.isfinite(D[i]) & (D[i] > 0)]
        if len(reachable):
            scores[i] = (len(reachable) / (n - 1)) * (len(reachable) / reachable.sum())
    return scores


def eigenvector_power(A, tolerance=1e-12, max_iterations=10_000):
    """Dominant eigenvector of the shared-hyperedge-weighted projection."""
    n = len(A)
    x = np.ones(n) / np.sqrt(n)
    for _ in range(max_iterations):
        y = A @ x
        norm = np.linalg.norm(y)
        if norm == 0:
            return np.zeros(n)
        y /= norm
        if min(np.linalg.norm(y - x), np.linalg.norm(y + x)) < tolerance:
            x = y
            break
        x = y
    return np.abs(x) / np.max(np.abs(x))


def betweenness_brandes(A):
    """Normalized vertex betweenness for an undirected, unweighted projection."""
    n = len(A)
    neighbors = adjacency_lists(A)
    centrality = np.zeros(n)
    for source in range(n):
        stack = []
        predecessors = [[] for _ in range(n)]
        shortest_count = np.zeros(n)
        shortest_count[source] = 1.0
        distance = np.full(n, -1, dtype=int)
        distance[source] = 0
        queue = deque([source])
        while queue:
            v = queue.popleft()
            stack.append(v)
            for w in neighbors[v]:
                if distance[w] < 0:
                    queue.append(w)
                    distance[w] = distance[v] + 1
                if distance[w] == distance[v] + 1:
                    shortest_count[w] += shortest_count[v]
                    predecessors[w].append(v)
        dependency = np.zeros(n)
        while stack:
            w = stack.pop()
            for v in predecessors[w]:
                dependency[v] += (
                    shortest_count[v] / shortest_count[w]
                ) * (1.0 + dependency[w])
            if w != source:
                centrality[w] += dependency[w]
    centrality /= 2.0  # undirected pairs were encountered from both directions
    if n > 2:
        centrality *= 2.0 / ((n - 1) * (n - 2))
    return centrality


def hypergraph_dangling(H):
    """Relative communication loss after removing each vertex's incidences."""
    base_D = distance_matrix(projected_adjacency(H, weighted=False))
    base_strength = communication_strength(base_D)
    scores = np.zeros(H.shape[0])
    modified_strengths = np.zeros(H.shape[0])
    for i in range(H.shape[0]):
        H_modified = H.copy()
        H_modified[i, :] = 0
        A_modified = projected_adjacency(H_modified, weighted=False)
        modified_strengths[i] = communication_strength(distance_matrix(A_modified))
        scores[i] = (base_strength - modified_strengths[i]) / base_strength
    return scores, base_strength, modified_strengths


def benchmark(function, repetitions=500):
    times = []
    result = None
    for _ in range(repetitions):
        start = perf_counter()
        result = function()
        times.append((perf_counter() - start) * 1_000)
    return result, median(times)


def main():
    vertices = sorted(PEOPLE)
    H = incidence_matrix(vertices, GROUPS)
    A_binary = projected_adjacency(H, weighted=False)
    A_weighted = projected_adjacency(H, weighted=True)

    degree, t_degree = benchmark(lambda: hyperdegree(H))
    close, t_close = benchmark(lambda: closeness(A_binary))
    eigen, t_eigen = benchmark(lambda: eigenvector_power(A_weighted))
    between, t_between = benchmark(lambda: betweenness_brandes(A_binary))
    dangling_result, t_dangling = benchmark(lambda: hypergraph_dangling(H))
    dangling, base_strength, modified_strengths = dangling_result

    print("Incidence matrix H:\n", H)
    print(f"\nBaseline communication strength: {base_strength:.4f}\n")
    header = (
        "Vertex", "Social role", "Hyperdegree", "Closeness", "Eigenvector",
        "Betweenness", "Modified strength", "Dangling centrality",
    )
    rows = []
    for index, vertex in enumerate(vertices):
        rows.append((
            f"v{vertex}", PEOPLE[vertex], degree[index], close[index], eigen[index],
            between[index], modified_strengths[index], dangling[index],
        ))

    print(" | ".join(header))
    for row in rows:
        print(" | ".join(
            [row[0], row[1]] + [f"{value:.4f}" for value in row[2:]]
        ))

    timing_rows = [
        ("Hyperdegree", t_degree, "O(sum |e|)"),
        ("Closeness", t_close, "O(n(n + m_p))"),
        ("Eigenvector power iteration", t_eigen, "O(k m_p)"),
        ("Betweenness (Brandes)", t_between, "O(n(n + m_p))"),
        (
            "Hypergraph Dangling (naive)", t_dangling,
            "O(n sum|e|^2 + n^2(n + m_p))",
        ),
    ]
    print("\nMedian runtime over 500 repetitions")
    for name, milliseconds, complexity in timing_rows:
        print(f"{name}: {milliseconds:.6f} ms; {complexity}")

    with open("hypergraph_centrality_results.csv", "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(rows)

    print("\nSaved: hypergraph_centrality_results.csv")


if __name__ == "__main__":
    main()


## Download the generated results


In [ ]:
from google.colab import files
files.download("hypergraph_centrality_results.csv")
